<a href="https://colab.research.google.com/github/sumitchhipa/python-learning/blob/main/Copy_of_faiss_bm_25.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install faiss-cpu rank_bm25 sentence-transformers google-generativeai

In [ ]:
import google.generativeai as genai
genai.configure(api_key="")

model = genai.GenerativeModel("gemini-2.5-flash")

In [ ]:
# step-3:: data
documents = [
    "Revenue growth in 2024 increased by 20 percent due to AI adoption.",
    "Profit is calculated as revenue minus expenses.",
    "In 2023, the company faced losses due to high operational cost.",
    "AI is transforming finance and business analytics.",
    "Revenue in 2024 is higher than 2023 significantly.",
    "Expenses include rent, salary, and infrastructure cost.",
    "Growth strategy includes AI and automation in 2024.",
]

In [ ]:
# STEP 4: Chunking
def chunk_text(docs, chunk_size=1):
    return docs

In [ ]:
# STEP 5: Embeddings
from sentence_transformers import SentenceTransformer

embed_model = SentenceTransformer('all-MiniLM-L6-v2')

chunks = chunk_text(documents)
embeddings = embed_model.encode(chunks)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
# STEP 6: FAISS Index
import faiss
import numpy as np
dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(np.array(embeddings))

In [ ]:
# STEP 7: BM25 (Keyword Search)
from rank_bm25 import BM25Okapi
tokenized_corpus = [doc.split() for doc in chunks]
bm25 = BM25Okapi(tokenized_corpus)

In [ ]:
# STEP 8: CACHE
cache = {}

In [ ]:
# STEP 9: Hybrid Search + Re-ranking
def hybrid_search(query, top_k=5):
    # FAISS Search
    query_vec = embed_model.encode([query])
    D, I = index.search(np.array(query_vec), top_k)
    faiss_results = [chunks[i] for i in I[0]]

    # BM25 Search
    tokenized_query = query.split()
    bm25_scores = bm25.get_scores(tokenized_query)
    bm25_results = [chunks[i] for i in np.argsort(bm25_scores)[::-1][:top_k]]
    # Combine (Hybrid)
    combined = list(set(faiss_results + bm25_results))
    return combined

In [ ]:
# STEP 10: Re-ranking
from sklearn.metrics.pairwise import cosine_similarity
def rerank(query, docs, top_k=3):
    query_vec = embed_model.encode([query])
    doc_vecs = embed_model.encode(docs)
    scores = cosine_similarity(query_vec, doc_vecs)[0]
    ranked = sorted(zip(docs, scores), key=lambda x: x[1], reverse=True)
    return [doc for doc, score in ranked[:top_k]]

In [ ]:
# STEP 11: Gemini Response
def generate_answer(query, context):
    prompt = f"""
    You are a financial AI assistant.
    Use the context below to answer the question.
    Context:
    {context}
    Question:
    {query}
    Answer:
    """
    response = model.generate_content(prompt)
    return response.text


In [ ]:
# STEP 12: Full Pipeline
def rag_pipeline(query):
    # Cache check
    if query in cache:
        print("From Cache")
        return cache[query]

    # Hybrid Search
    retrieved_docs = hybrid_search(query)

    # Re-ranking
    top_docs = rerank(query, retrieved_docs)

    # Context
    context = "\n".join(top_docs)


    # Gemini call
    answer = generate_answer(query, context)

    # Store in cache
    cache[query] = answer

    return answer


In [ ]:
# STEP 13: Test
query = "What is revenue growth in 2024?"
response = rag_pipeline(query)
print(response)

Revenue growth in 2024 increased by 20 percent.


In [ ]:
# STEP 14: Cache Test
# Re-Run the same query
response = rag_pipeline(query)
print(response)

From Cache
Revenue growth in 2024 increased by 20 percent.


In [ ]:
cache

{'What is revenue growth in 2024?': 'Revenue growth in 2024 increased by 20 percent.'}